# 06 - Motion Planning

## What / Why / How

**What we are trying to do:** Plan collision-free motion with graph search and sampling-based planning.

**Why this matters:** A controller can follow commands, but a planner decides which route is safe and feasible before motion begins.

**How we will do it:** Implement A* on a grid, visualize the path, then grow an RRT through continuous 2D space with obstacle checks.

## Goal

Motion planning answers: how can a robot move from start to goal without collisions?

You will implement:

- A* on a grid
- Collision checking
- A small RRT planner in continuous 2D

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## A* Grid Planner

In [ ]:
import heapq

H, W = 20, 24
grid = np.zeros((H, W), dtype=int)
grid[5:16, 10] = 1
grid[4, 4:14] = 1
grid[13, 10:21] = 1
start = (2, 2)
goal = (17, 21)

def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def neighbors(cell):
    r, c = cell
    for dr, dc in [(1,0), (-1,0), (0,1), (0,-1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < H and 0 <= nc < W and grid[nr, nc] == 0:
            yield (nr, nc)

def astar(start, goal):
    frontier = [(0, start)]
    came_from = {start: None}
    cost_so_far = {start: 0}
    while frontier:
        _, current = heapq.heappop(frontier)
        if current == goal:
            break
        for nxt in neighbors(current):
            new_cost = cost_so_far[current] + 1
            if nxt not in cost_so_far or new_cost < cost_so_far[nxt]:
                cost_so_far[nxt] = new_cost
                priority = new_cost + heuristic(nxt, goal)
                heapq.heappush(frontier, (priority, nxt))
                came_from[nxt] = current
    if goal not in came_from:
        return []
    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = came_from[cur]
    return path[::-1]

path = astar(start, goal)
print("path length:", len(path))
print("first five cells:", path[:5])

In [ ]:
if HAS_PLOT:
    img = grid.astype(float)
    for r, c in path:
        img[r, c] = 0.5
    plt.figure(figsize=(7, 4))
    plt.imshow(img, origin="lower", cmap="gray_r")
    plt.scatter([start[1], goal[1]], [start[0], goal[0]], c=["tab:green", "tab:red"], s=80)
    plt.title("A* path")
    plt.show()
else:
    plot_unavailable()

## RRT In Continuous Space

In [ ]:
rng = np.random.default_rng(4)
rects = [(3, 3, 2, 5), (6, 0, 1.5, 5), (7, 6, 2, 2)]

def in_collision(point):
    x, y = point
    if x < 0 or x > 10 or y < 0 or y > 10:
        return True
    for rx, ry, rw, rh in rects:
        if rx <= x <= rx + rw and ry <= y <= ry + rh:
            return True
    return False

def segment_free(a, b, checks=20):
    for alpha in np.linspace(0, 1, checks):
        p = (1 - alpha) * a + alpha * b
        if in_collision(p):
            return False
    return True

def rrt(start, goal, step_size=0.45, iterations=2500):
    nodes = [np.array(start, dtype=float)]
    parents = [-1]
    for _ in range(iterations):
        sample = np.array(goal) if rng.random() < 0.1 else rng.uniform(0, 10, size=2)
        dists = [np.linalg.norm(n - sample) for n in nodes]
        nearest_idx = int(np.argmin(dists))
        direction = sample - nodes[nearest_idx]
        norm = np.linalg.norm(direction)
        if norm < 1e-9:
            continue
        new = nodes[nearest_idx] + step_size * direction / norm
        if segment_free(nodes[nearest_idx], new):
            nodes.append(new)
            parents.append(nearest_idx)
            if np.linalg.norm(new - goal) < 0.6 and segment_free(new, np.array(goal)):
                nodes.append(np.array(goal, dtype=float))
                parents.append(len(nodes) - 2)
                break
    idx = len(nodes) - 1
    if np.linalg.norm(nodes[idx] - goal) > 1e-6:
        return nodes, parents, []
    route = []
    while idx != -1:
        route.append(nodes[idx])
        idx = parents[idx]
    return nodes, parents, route[::-1]

nodes, parents, route = rrt([0.8, 0.8], [9.2, 9.0])
print("nodes:", len(nodes), "route points:", len(route))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 5))
    for rx, ry, rw, rh in rects:
        plt.gca().add_patch(plt.Rectangle((rx, ry), rw, rh, color="black", alpha=0.25))
    for i, parent in enumerate(parents):
        if parent >= 0:
            a, b = nodes[i], nodes[parent]
            plt.plot([a[0], b[0]], [a[1], b[1]], color="gray", alpha=0.15)
    if route:
        route_arr = np.array(route)
        plt.plot(route_arr[:, 0], route_arr[:, 1], color="tab:red", linewidth=3)
    plt.xlim(0, 10)
    plt.ylim(0, 10)
    plt.grid(True, alpha=0.2)
    plt.title("RRT")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Change the A* heuristic and observe path length.
2. Add diagonal neighbors and compare.
3. Add a narrow passage to the RRT world.
4. Add path smoothing after RRT.